# **LLM for Social Simulation — API Basics and Introduction**

> This notebook introduces the **fundamentals of using APIs and simple model calls**.

Run the following code to install the required dependencies.




In [ ]:
!pip install openai
!pip install huggingface_hub
!pip install retrying

**After adding your access token to the Colab secrets, run the code below to check if it has been loaded successfully.**

> **Note:** If you modify your token, you need to rerun the code below to reload your new token.


In [ ]:
from google.colab import userdata
import os

hf_token = userdata.get('HF_TOKEN')

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("✅ HF_TOKEN loaded:", hf_token[:8] + "...")
    print("✅ Environment variable set:", "HF_TOKEN" in os.environ)
else:
    print("❌ HF_TOKEN not found")


### **Part 1: Getting Started with Large Language Model APIs**

**1. Model Selection**

Hugging Face offers a wide range of language models.  
You can easily switch models by replacing the model name in your code.  
Below are examples of the latest models available on Hugging Face:

- `MiniMaxAI/MiniMax-M2`  
- `deepseek-ai/DeepSeek-V3.2-Exp`  
- `deepseek-ai/DeepSeek-V3`  
- `moonshotai/Kimi-K2-Thinking`

For more model options, visit:  
👉 [https://huggingface.co/models](https://huggingface.co/models)

> ⚠️ Note: long-thinking models such as **Kimi-K2-Thinking** tend to consume more credits.


**2. Temperature**

Controls the randomness of the model's output. Lower values make responses more deterministic and focused, while higher values allow for more creative and diverse outputs. Use low values for factual tasks and high values for creative tasks.

**3. Top P**

This is nucleus sampling, where only tokens from the top probability mass (up to `top_p`) are considered. Lower values encourage more focused responses, while higher values increase the diversity of possible outputs.

**4. Max Tokens**

This limits the total number of tokens the model can generate, helping control response length and prevent irrelevant output.


> Run the code in the cell below to call the LLM through the Hugging Face API.

In [ ]:
from openai import OpenAI
import os

# Connect to Hugging Face router using your HF token
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"]
)

# Generate a chat completion
completion = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-V3.2-Exp",  # Model selection (choose from HF models)
    messages=[{"role": "user", "content": "Hello!"}],
    temperature=0.3,  # How random the answer is (lower = more exact, higher = more creative)
    top_p=0.9,        # How wide the model’s word choices are (lower = safer, higher = freer)
    max_tokens=100    # Max length of the reply (in model “words”)
)

print(completion.choices[0].message.content)


**Structured Output Prompting**

In real-world applications, we often want the model to return **structured data** —  
for example, JSON, tables, or key-value fields — instead of plain natural language text.  

This technique is called **Structured Output Prompting**,  
which explicitly instructs the model to format its response in a predefined structure.

In [ ]:
# === Structured JSON Output Example ===
json_prompt = """You are a helpful assistant.
Summarize this text in one sentence, then Return RAW JSON only.

Text: Artificial Intelligence enables machines to learn from data and perform tasks that typically require human intelligence.

Output only JSON in this format:
{"summary": "<your concise summary>"}"""

completion_json = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-V3.2-Exp",   # or other models
    messages=[{"role": "user", "content": json_prompt}],
    temperature=0.3,
    top_p=0.9,
    max_tokens=150
)

print(completion_json.choices[0].message.content)


### **Part 2: Prompt Engineering Basics**


> **Goal:** Clearly define the task, constraints, and output format to make downstream code simpler.

Before we begin, please run the code in the cell below to set up a helper function for the upcoming prompt demonstrations.


In [ ]:
def hf_call(prompt, model="deepseek-ai/DeepSeek-V3",
            temperature=0.3, top_p=0.9, max_tokens=100):
    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens
    )
    return completion.choices[0].message.content


**1. Zero-Shot Prompting**

Large language models (LLMs) today, such as GPT-5, Claude Sonnet 4.5, and Gemini 2.5 Pro, are tuned to follow instructions and are trained on large amounts of data. Large-scale training makes these models capable of performing some tasks in a "zero-shot" manner. Zero-shot prompting means that the prompt used to interact with the model won't contain examples or demonstrations. The zero-shot prompt directly instructs the model to perform a task without any additional examples to steer it.

In [ ]:
zero_shot_prompt = """You are a sentiment classifier.
Classify the sentiment of the following text as Positive, Negative, or Neutral.

Text: I think the vacation is okay.
Output format:
Sentiment: <label>"""

print(hf_call(zero_shot_prompt))


**2. Few-Shot Prompting**

While large-language models demonstrate remarkable zero-shot capabilities, they still fall short on more complex tasks when using the zero-shot setting. Few-shot prompting can be used as a technique to enable in-context learning where we provide demonstrations in the prompt to steer the model to better performance. The demonstrations serve as conditioning for subsequent examples where we would like the model to generate a response.

According to [Touvron et al. 2023](https://arxiv.org/pdf/2302.13971.pdf) few shot properties first appeared when models were scaled to a sufficient size ([Kaplan et al., 2020](https://arxiv.org/abs/2001.08361)).

In [ ]:
few_shot_prompt = """You are a sentiment classifier.
Classify the sentiment of the last sentence only, based on the examples.

Examples:
This is awesome! // Positive
This is bad! // Negative
Wow that movie was rad! // Positive

Now classify this one:
What a horrible show!

Output format:
Sentiment: <label>"""

print(hf_call(few_shot_prompt))


**3. Chain-of-Thought Prompting**

Introduced in [Wei et al. (2022)](https://arxiv.org/abs/2201.11903), chain-of-thought (CoT) prompting enables complex reasoning capabilities through intermediate reasoning steps. You can combine it with few-shot prompting to get better results on more complex tasks that require reasoning before responding.

![](https://www.promptingguide.ai/_next/image?url=%2F_next%2Fstatic%2Fmedia%2Fcot.1933d9fe.png&w=1920&q=75)  

In [ ]:
cot_prompt = """You are a reasoning assistant.
Solve the problem step by step in at most 3 short steps, then give the final answer on a new line as: Answer: <number>.

Question:
If there are 3 cars and each car has 4 wheels, and there are 2 spare wheels, how many total wheels are there?

Let's think step by step."""
print(hf_call(cot_prompt, max_tokens=256))


**4. Structured Output Prompting**

In **Part 1**, we briefly demonstrated a simple example where the model summarized a text into JSON format.  

Here, we will **revisit and expand that idea**, turning it into a complete **Prompt Engineering technique**.  
Structured outputs are not just about controlling format — they teach the model to comply with a **data schema**,  
which is crucial for connecting LLMs to downstream systems.


In [ ]:
structured_prompt = """You are an information extractor.
Read the text carefully, extract all relevant details, and return RAW JSON only.

Text:
Alice bought 3 apples for $5 and 2 oranges for $4.

Output only JSON in this format:
{"buyer":"","items":[{"name":"","quantity":0,"price":0}],"total_cost":0}
"""

print(hf_call(structured_prompt, max_tokens=200, temperature=0.2))


### 🧠 Technique Notes
- Use explicit instructions such as *“Output only JSON”* to prevent narrative responses.  
- Provide a **clear schema example** (e.g., fields like `buyer`, `items`, `total_cost`).  
- Validate the output using `json.loads()` to ensure compliance.  
- This approach enables **programmable outputs** — an essential foundation  
  for later modules on **social simulation**, where agents need to exchange structured outputs rather than free text.

In [ ]:
import re, json

output = hf_call(structured_prompt, max_tokens=200).strip()

m = re.search(r"```(?:json)?\s*(.*?)```", output, re.S|re.I)
if m:
    output = m.group(1).strip()

try:
    parsed = json.loads(output)
    print("✅ Valid JSON detected:")
    print(parsed)
except json.JSONDecodeError:
    print("⚠️ Model output is not valid JSON:")
    print(output)


### **Using the Shubiaobiao API in Colab**

**After** obtaining your **Shubiaobiao API key**, add it to your **Colab Secrets** (under *“secrets” → “+ New Secret”*).  
Once the key is saved, you can use the following code to call an LLM through the **Shubiaobiao API**.


In [6]:
# from openai import OpenAI
# import os

# # Connect to Shubiaobiao API instead of Hugging Face
# client = OpenAI(
#     base_url="https://api.shubiaobiao.com/v1",  # ✅ Shubiaobiao base URL
#     api_key=os.environ["yourkey"]            # ✅ Shubiaobiao API key (set in Colab Secret)
# )

# # Generate a chat completion
# completion = client.chat.completions.create(
#     model="gpt-4o",  # ✅ Model name from Shubiaobiao
#     messages=[
#         {"role": "system", "content": "You are a helpful assistant."},
#         {"role": "user", "content": "Hello!"}
#     ]
# )

# # Print the response
# print(completion.choices[0].message.content)


KeyError: 'sk-7MPDKIdGJngMDr5i5a15907dF4Ec4c8294AfB58dAe43Db5e'